In [2]:
%load_ext autoreload
%autoreload 2
from IPython.core.display import display, HTML
display(HTML("<style>.container { width:95% !important; }</style>"))
import warnings
warnings.filterwarnings("ignore")

import os
import pandas as pd
import sys
sys.path.insert(0, '/home/kat/Repos/SALSA/')

import deepchem as dc

### Load HIV dataset, clean smiles, save it ... 

In [3]:
tokens_salsa = '#%()+-0123456789<=>BCFHILNOPRSX[]cnos'
tokens_hiv = 'Ub)gsP5u=8H[d]noW06%2GLlO+NFSTa9ZCeI-r4(1ihAVBRc3M7pt#'

In [4]:
%%capture
_, datasets, _ = dc.molnet.load_hiv()
ds_train, ds_valid, ds_test = datasets

In [4]:
def undo_BrCl_singles(smi):
    smi = smi.replace('R','Br')
    return smi.replace('L','Cl')
def do_BrCl_singles(smi):
    smi = smi.replace('Br','R')
    return smi.replace('Cl','L')   

from rdkit import RDLogger  
RDLogger.DisableLog('rdApp.*')
from utilities.rdkit_utils import *
from tqdm.notebook import tqdm

cuts = ['train', 'test'] #,'val','test']

bad_smis = []
all_smis = []
raw_smis = []

for i,cut in enumerate(cuts):
    
    ds = datasets[i]
    X_and_ys = zip(ds.ids, ds.y)
    
    smis = []
    labs = []
    
    for smi,lab in tqdm(X_and_ys, total=len(ds)):        
        _smi = get_cansmiles(smi)
        components = _smi.split('.')
        components = sorted(components, key=lambda x: len(x))
        smi = components[-1]
        
        if smi:
            raw_smis.append(smi)
            
            # Need to match SALSA's vocab !!!
            smi = do_BrCl_singles(smi)  
            
            if not all(c in tokens_salsa for c in smi):
                bad_smis.append(smi)
                if 'U' in smi:
                    print('---\n',smi)
                    
            elif len(smi) > 120:
                bad_smis.append(smi)
            else:
                smis.append(smi)
                all_smis.append(smi)
                labs.append(lab)
            
    df = pd.DataFrame(smis,columns=['smiles'])
    df.to_csv(f'data/model_ready/hiv/{cut}/anchor_smiles.csv',index=False)
    
    df = pd.DataFrame(labs,columns=['y'])
    df.to_csv(f'data/model_ready/hiv/{cut}/labels.csv',index=False)

  0%|          | 0/32901 [00:00<?, ?it/s]

  0%|          | 0/4113 [00:00<?, ?it/s]

---
 COc1cccc2c1[OH+][U]134(=O)(=O)(O)(O)([O+]=C(c5ccncc5)[N-][N+]1=C2)[O+]=C(c1ccncc1)[N-][N+]3=Cc1cccc(OC)c1[OH+]4
---
 COc1cccc2c1[OH+][U-5](=O)(=O)(O)(O)[O+]=C2
---
 COc1cccc2c1[OH+][U+2]1(=O)(=O)(O)(O)(O)[O+]=C(N)[N-][N+]1=C2


### Look at the HIV tokens not in SALSA vocab ...

In [5]:
print(len(bad_smis), len(all_smis), len(raw_smis))

v = list(set(''.join(all_smis)))
print(''.join(v))

len(raw_smis)-len(bad_smis)

1403 35611 37014
6SF(3n5I]=7Ps)L+B-1[c9CH0Oo%2R#48N


35611

In [6]:
bad_tokens = list(set(list(tokens_hiv)) - set(tokens_salsa))
# bad_tokens = list(set(list(tokens_salsa)) ^ set(tokens_hiv))
print(len(bad_tokens))
print(bad_tokens)

20
['G', 'a', 'A', 'T', 'b', 'V', 'l', 'u', 'M', 'd', 't', 'g', 'r', 'h', 'U', 'e', 'Z', 'p', 'W', 'i']


### Load back in the cleaned HIV dataset.

In [7]:
df_hiv = pd.read_csv('data/model_ready/hiv/train/anchor_smiles.csv')
hiv_vocab = list(set(list(''.join(df_hiv.smiles.values))))
print(len(hiv_vocab))
print(len(tokens_salsa))

34
37
